# 01 - Database Connection

The goal of this notebook is to connect to ABC Grocery's PostgreSQL database and set up what the SQL agent will be allowed to see.

I build the connection from credentials in `.env`, check it works, and wrap it in LangChain's `SQLDatabase`, limited to the two tables the agent needs. Then I look at exactly what the agent will see about each table.



## Setup

The database credentials and OpenAI key are loaded from `.env`. Settings that aren't secret (schema, tables, sample rows, timeout) come from `src/config.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
from src import config
import os
from dotenv import load_dotenv
load_dotenv()

import sqlalchemy as sa
from langchain_community.utilities import SQLDatabase

## Check the credentials loaded

If a variable is missing from `.env`, the connection string would contain `None` and the error would be hard to read. I check that each one is set, without printing the values.

In [ ]:
for name in ["POSTGRES_USER", "POSTGRES_PASSWORD", "POSTGRES_HOST", "POSTGRES_PORT", "POSTGRES_DBNAME", "OPENAI_API_KEY"]:
    status = "set" if os.getenv(name) else "MISSING"
    print(f"{name}: {status}")

POSTGRES_USER: set
POSTGRES_PASSWORD: set
POSTGRES_HOST: set
POSTGRES_PORT: set
POSTGRES_DBNAME: set
OPENAI_API_KEY: set


## Connection string

The connection string is built from the `.env` variables. `sslmode=require` means the connection to the cloud database is encrypted.

In [ ]:
POSTGRES_URI = (f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
                f"@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DBNAME')}?sslmode=require")


## Engine and health check

- **`pool_pre_ping=True`:** checks a pooled connection is still alive before using it, so a connection the cloud database closed while idle doesn't cause a failed query.
- **`statement_timeout`:** any query running longer than 15 seconds is cancelled, so a slow query can't hang the agent.
- **`select 1`:** `create_engine` doesn't connect to anything, it only stores the settings. This tiny query forces a real connection, so a wrong password or network problem shows up straight away. I tested this by changing the password in `.env`: the error appeared at `engine.connect()`.

In [ ]:
# create the database engine
engine = sa.create_engine(POSTGRES_URI,
                          pool_pre_ping=True,
                          connect_args={"options": f"-c statement_timeout={config.STATEMENT_TIMEOUT_MS}"} 
                          ) 

# check the connection
with engine.connect() as conn:
    conn.exec_driver_sql("select 1")
    timeout = conn.exec_driver_sql("show statement_timeout").scalar()

print("Connection OK")
print("Statement timeout:", timeout)

Connection OK
Statement timeout: 15s


## SQLDatabase wrapper

`SQLDatabase` is what the agent's tools use to talk to the database. It's limited to the `grocery_db` schema and only two tables, so the agent can't see or query anything else.

In [ ]:

db = SQLDatabase(engine=engine,
                 schema=config.SCHEMA_NAME,
                 include_tables=config.TABLES,
                 sample_rows_in_table_info=config.SAMPLE_ROWS)

print("Usable tables:", db.get_usable_table_names())


Usable tables: ['customer_details', 'transactions']


## What the agent sees

`get_table_info()` returns the text the agent receives about each table: the table structure plus 5 sample rows. The samples show the agent what real values look like, for example that gender is stored as 'M' and 'F'.

In [ ]:
print(db.get_table_info())


CREATE TABLE grocery_db.customer_details (
	customer_id INTEGER, 
	distance_from_store NUMERIC(10, 2), 
	gender VARCHAR(2), 
	credit_score NUMERIC(3, 2)
)

/*
5 rows from customer_details table:
customer_id	distance_from_store	gender	credit_score
74	3.38	F	0.59
524	4.76	F	0.52
607	4.45	F	0.49
343	0.91	M	0.54
322	3.02	F	0.63
*/


CREATE TABLE grocery_db.transactions (
	customer_id INTEGER, 
	transaction_date DATE, 
	transaction_id BIGINT, 
	product_area_id INTEGER, 
	num_items INTEGER, 
	sales_cost NUMERIC(10, 2)
)

/*
5 rows from transactions table:
customer_id	transaction_date	transaction_id	product_area_id	num_items	sales_cost
1	2020-04-10	435657533999	3	7	19.16
1	2020-04-10	435657533999	2	5	7.71
1	2020-06-02	436189770685	4	4	26.97
1	2020-06-02	436189770685	1	2	38.52
1	2020-06-10	436265380298	4	4	22.13
*/


## Summary

- Connected to the PostgreSQL database using credentials from `.env`. The connection string is built in code but never printed, because it contains the password.
- **Health check:** `select 1` runs, so the credentials and network access work. Testing with a wrong password showed that `create_engine` doesn't connect; the first real connection happens at `engine.connect()`, which is why the health check is needed. The statement timeout is set to 15 seconds.
- **Scoping:** `SQLDatabase` only exposes `customer_details` and `transactions` from the `grocery_db` schema.
- **What the agent sees:** each table's structure plus 5 sample rows. The samples show that one transaction can have several rows (one per product area), so counting and summing transactions must be done carefully.
- The database stores `transaction_id` as `BIGINT`, while the system prompt says `INT`. The agent sees the real type from the database.

**Next:** move the connection code into `src/database.py`, then build the agent in notebook 02.